# Clinical Document Intelligence — Pipeline Walkthrough

This notebook demonstrates the standalone pipeline (`pipeline/`) described
in the accompanying paper: given a real clinical document, it shows the
full path from raw text to a grounding-validated (or correctly-rejected)
safety flag.

Two examples are shown, both drawn directly from this paper's evaluation:

1. **A genuinely accepted flag** — a verbatim, contiguous quote that
   correctly grounds a real clinical concern.
2. **A genuinely rejected flag** — the paper's flagship
   *composition-fabrication* example: a quote that stitches two
   individually-real but non-adjacent sentence fragments into a claim
   the source document never actually makes.

Both examples use the real `grounding.py` module from this repository —
no simulation, no mocked results.


## Setup

In [ ]:
import sys
sys.path.insert(0, ".")  # so grounding.py / extraction.py are importable

from grounding import check_contiguous_grounding, run_four_guards


## Example 1: A genuinely grounded flag (accepted)

This is drawn directly from the paper's MTSamples generalisation check.
The source document genuinely, verbatim contains this dose statement —
notice it survives Guard 3's calibration fix (short quotes must be fully
self-contiguous), which this exact case motivated.

In [ ]:
source_document_text = "3. Metformin 1000 mg p.o. b.i.d."
proposed_quote = "Metformin 1000 mg"

result = check_contiguous_grounding(proposed_quote, source_document_text)
print(f"Quote:    {proposed_quote!r}")
print(f"Verdict:  {result['verdict']}")
print(f"Passes:   {result['passes']}")
print(f"Longest contiguous run: {result['longest_contiguous_run']} tokens")


## Example 2: A composition-fabricated flag (correctly rejected)

This is the paper's flagship example of *composition-fabrication*: a
quote that has **complete token overlap** with the source document —
every individual word genuinely appears somewhere in the text — but
stitches together two non-adjacent sentence fragments into a claim the
document never actually makes.

A naive grounding check based only on token overlap would accept this.
Guard 3 requires the quote's tokens to appear as a **genuinely
contiguous span**, and correctly rejects it.

In [ ]:
source_document_text = (
    "Patient reports symptoms consistent with NYHA class II. "
    "Plan: Continue current heart failure therapy."
)
fabricated_quote = "NYHA class II consistent with heart failure therapy"

result = check_contiguous_grounding(fabricated_quote, source_document_text)
print(f"Source document: {source_document_text!r}")
print()
print(f"Fabricated quote: {fabricated_quote!r}")
print(f"Verdict:  {result['verdict']}")
print(f"Passes:   {result['passes']}")
print(f"Token overlap ratio: {result['overlap_ratio']:.2f}  (high! every word is real)")
print(f"Longest contiguous run: {result['longest_contiguous_run']} tokens  (low — the words are not actually adjacent)")
print()
print("Despite near-total token overlap, Guard 3 correctly rejects this quote")
print("because the words never actually appear together, in this order, in the source.")


## Example 3: Full four-guard orchestration on a proposed flag

This shows `run_four_guards`, the same function `run_pipeline.py` uses
end-to-end, applied to a complete proposed-flag dict (as an LLM would
produce), not just a bare quote.

In [ ]:
proposed_flag = {
    "cited_document_id": "example_doc",
    "source_quote": "NYHA class II consistent with heart failure therapy",
    "category": "AI_TEST_EXAMPLE",
    "description": "NYHA class II consistent with heart failure therapy",
}

verdict = run_four_guards(
    proposed_flag,
    cited_document_text=source_document_text,
    valid_document_ids={"example_doc"},
)

print(f"Accepted: {verdict['accepted']}")
print(f"Failed at guard: {verdict['failed_guard']}")
print(f"Verdict: {verdict['verdict']}")


## Next steps

- Run this against your own PDF: see `pipeline/README.md` for the full
  `run_pipeline.py` setup (requires a Python 3.11 environment and your
  own Anthropic API key).
- Full test suite: `pytest pipeline/tests/ -v` (39 tests, covering both
  examples above plus additional Guard 1/2/4 cases).
- Full evaluation methodology and all 30 real-document results: see the
  accompanying paper, Evaluation section.
